# FT-Transformer (Feature Tokenizer + Transformer)

From "Revisiting Deep Learning Models for Tabular Data" (Gorishniy et al., 2021).

Each feature — numeric or categorical — is **tokenized** into a d_token-dimensional vector.
All tokens (plus a prepended [CLS] token) pass through a standard Transformer encoder.
The [CLS] token's final representation is used for binary classification.

**Architecture**:
- `NumericalTokenizer`: per-feature `Linear(1 → d_token)` (separate weights per feature)
- `CategoricalTokenizer`: per-feature `Embedding(n_classes → d_token)`
- `TransformerEncoder`: n_blocks × pre-norm (LayerNorm → MHA → LayerNorm → FFN)
- `Head`: LayerNorm → Linear(d_token → 1)

**Baselines to beat**:
- CatBoost default single seed:  CV AUC = 0.95533
- CatBoost seed-avg (10 seeds):  CV AUC = 0.95540
- TabNet (1×5):                  CV AUC = 0.95369

## Imports & Data

In [1]:
import math
import subprocess
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__}  |  Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

KAGGLE_DATA = Path('/kaggle/input/playground-series-s6e2')
LOCAL_DATA  = Path('data')
DATA_DIR    = KAGGLE_DATA if KAGGLE_DATA.exists() else LOCAL_DATA

def prep(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower().str.replace(r'\s+', '_', regex=True)
    if 'heart_disease' in df.columns:
        df['heart_disease'] = df['heart_disease'].map({'Absence': 0, 'Presence': 1})
    return df

train = prep(pd.read_csv(DATA_DIR / 'train.csv'))
test  = prep(pd.read_csv(DATA_DIR / 'test.csv'))
ss    = pd.read_csv(DATA_DIR / 'sample_submission.csv')

FEATURES     = [c for c in train.columns if c not in ['heart_disease', 'id']]
CAT_FEATURES = ['sex', 'chest_pain_type', 'fbs_over_120', 'ekg_results',
                'exercise_angina', 'slope_of_st', 'number_of_vessels_fluro', 'thallium']
NUM_FEATURES = [f for f in FEATURES if f not in CAT_FEATURES]

y = train['heart_disease'].values
print(f'Train: {train.shape}  Test: {test.shape}')
print(f'Numeric  ({len(NUM_FEATURES)}): {NUM_FEATURES}')
print(f'Categorical ({len(CAT_FEATURES)}): {CAT_FEATURES}')

PyTorch 2.0.1+cu118  |  Device: cuda
GPU: NVIDIA GeForce RTX 3060
Train: (630000, 15)  Test: (270000, 14)
Numeric  (5): ['age', 'bp', 'cholesterol', 'max_hr', 'st_depression']
Categorical (8): ['sex', 'chest_pain_type', 'fbs_over_120', 'ekg_results', 'exercise_angina', 'slope_of_st', 'number_of_vessels_fluro', 'thallium']


## Preprocessing

- Numeric: `StandardScaler` (FT-Transformer's linear tokenizer benefits from normalized inputs)
- Categorical: `OrdinalEncoder` → integer indices for embedding lookup

In [2]:
scaler = StandardScaler()
enc    = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

X_num      = scaler.fit_transform(train[NUM_FEATURES].values).astype(np.float32)
X_num_test = scaler.transform(test[NUM_FEATURES].values).astype(np.float32)

X_cat      = enc.fit_transform(train[CAT_FEATURES].values).astype(np.int64)
X_cat_test = enc.transform(test[CAT_FEATURES].values).astype(np.int64)

# +1 for unknown token (OrdinalEncoder uses -1 for unknown, we'll clamp to 0 later)
CARDINALITIES = [int(len(cats)) + 1 for cats in enc.categories_]

# Clamp unknowns to last index (cardinality - 1)
for i, card in enumerate(CARDINALITIES):
    X_cat[:, i]      = np.clip(X_cat[:, i],      0, card - 1)
    X_cat_test[:, i] = np.clip(X_cat_test[:, i], 0, card - 1)

print(f'X_num shape: {X_num.shape}  X_cat shape: {X_cat.shape}')
print(f'Cardinalities: {dict(zip(CAT_FEATURES, CARDINALITIES))}')

X_num shape: (630000, 5)  X_cat shape: (630000, 8)
Cardinalities: {'sex': 3, 'chest_pain_type': 5, 'fbs_over_120': 3, 'ekg_results': 4, 'exercise_angina': 3, 'slope_of_st': 4, 'number_of_vessels_fluro': 5, 'thallium': 4}


## FT-Transformer Model

Clean PyTorch implementation following the original paper.

In [3]:
class NumericalTokenizer(nn.Module):
    """Per-feature linear projection: (B, n_num) -> (B, n_num, d_token)"""
    def __init__(self, n_features: int, d_token: int):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(n_features, d_token))
        self.bias   = nn.Parameter(torch.zeros(n_features, d_token))
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.weight[None] * x[:, :, None] + self.bias[None]


class CategoricalTokenizer(nn.Module):
    """Per-feature embedding lookup: (B, n_cat) int -> (B, n_cat, d_token)"""
    def __init__(self, cardinalities: list, d_token: int):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(n, d_token) for n in cardinalities
        ])
        for emb in self.embeddings:
            nn.init.kaiming_uniform_(emb.weight, a=math.sqrt(5))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.stack([emb(x[:, i]) for i, emb in enumerate(self.embeddings)], dim=1)


class FTTransformerBlock(nn.Module):
    """Pre-norm transformer block (as in the original paper)"""
    def __init__(self, d_token: int, n_heads: int, d_ffn: int,
                 attn_dropout: float = 0.2, ffn_dropout: float = 0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_token)
        self.attn  = nn.MultiheadAttention(d_token, n_heads,
                                           dropout=attn_dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_token)
        self.ffn   = nn.Sequential(
            nn.Linear(d_token, d_ffn),
            nn.GELU(),
            nn.Dropout(ffn_dropout),
            nn.Linear(d_ffn, d_token),
            nn.Dropout(ffn_dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        normed = self.norm1(x)
        x = x + self.attn(normed, normed, normed)[0]
        x = x + self.ffn(self.norm2(x))
        return x


class FTTransformer(nn.Module):
    def __init__(
        self,
        n_num: int,
        cardinalities: list,
        d_token: int = 192,
        n_blocks: int = 3,
        n_heads: int = 8,
        d_ffn: int = None,
        attn_dropout: float = 0.2,
        ffn_dropout: float = 0.1,
    ):
        super().__init__()
        if d_ffn is None:
            d_ffn = int(d_token * 4 / 3)  # paper default: 4/3 × d_token

        self.num_tok = NumericalTokenizer(n_num, d_token) if n_num > 0 else None
        self.cat_tok = CategoricalTokenizer(cardinalities, d_token) if cardinalities else None

        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_token))
        nn.init.trunc_normal_(self.cls_token, std=0.02)

        self.blocks = nn.ModuleList([
            FTTransformerBlock(d_token, n_heads, d_ffn, attn_dropout, ffn_dropout)
            for _ in range(n_blocks)
        ])

        self.norm = nn.LayerNorm(d_token)
        self.head = nn.Linear(d_token, 1)

    def forward(self, x_num: torch.Tensor, x_cat: torch.Tensor) -> torch.Tensor:
        parts = []
        if self.num_tok is not None:
            parts.append(self.num_tok(x_num))
        if self.cat_tok is not None:
            parts.append(self.cat_tok(x_cat))

        x   = torch.cat(parts, dim=1)                         # (B, n_feat, d)
        cls = self.cls_token.expand(x.size(0), -1, -1)
        x   = torch.cat([cls, x], dim=1)                      # (B, 1+n_feat, d)

        for block in self.blocks:
            x = block(x)

        return self.head(self.norm(x[:, 0])).squeeze(1)        # CLS logit


# Sanity-check param count
m_check = FTTransformer(
    n_num=len(NUM_FEATURES),
    cardinalities=CARDINALITIES,
    d_token=192, n_blocks=3, n_heads=8
)
n_params = sum(p.numel() for p in m_check.parameters())
print(f'FT-Transformer params: {n_params:,}  ({n_params/1e6:.2f}M)')
del m_check

FT-Transformer params: 751,873  (0.75M)


## Dataset & Training Utilities

In [4]:
class TabularDataset(Dataset):
    def __init__(self, x_num, x_cat, y=None):
        self.x_num = torch.from_numpy(x_num)
        self.x_cat = torch.from_numpy(x_cat)
        self.y     = torch.from_numpy(y.astype(np.float32)) if y is not None else None

    def __len__(self):
        return len(self.x_num)

    def __getitem__(self, idx):
        if self.y is not None:
            return self.x_num[idx], self.x_cat[idx], self.y[idx]
        return self.x_num[idx], self.x_cat[idx]


def make_loader(x_num, x_cat, y=None, batch_size=1024, shuffle=False):
    ds = TabularDataset(x_num, x_cat, y)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      pin_memory=(device.type == 'cuda'), num_workers=4)


def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0.
    for x_num, x_cat, y_batch in loader:
        x_num, x_cat, y_batch = x_num.to(device), x_cat.to(device), y_batch.to(device)
        optimizer.zero_grad()
        loss = F.binary_cross_entropy_with_logits(model(x_num, x_cat), y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item() * len(y_batch)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def predict_proba(model, loader):
    model.eval()
    preds = []
    for batch in loader:
        x_num, x_cat = batch[0].to(device), batch[1].to(device)
        preds.append(torch.sigmoid(model(x_num, x_cat)).cpu().numpy())
    return np.concatenate(preds)


def fit_model(model, x_num_tr, x_cat_tr, y_tr,
              x_num_val, x_cat_val, y_val,
              batch_size=1024, max_epochs=100, patience=15,
              lr=1e-4, weight_decay=1e-5):

    tr_loader  = make_loader(x_num_tr,  x_cat_tr,  y_tr,  batch_size, shuffle=True)
    val_loader = make_loader(x_num_val, x_cat_val, y_val, batch_size * 4)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr,
        steps_per_epoch=len(tr_loader),
        epochs=max_epochs
    )

    best_auc, best_state, no_improve = 0., None, 0

    for epoch in range(max_epochs):
        loss    = train_epoch(model, tr_loader, optimizer, scheduler)
        val_auc = roc_auc_score(y_val, predict_proba(model, val_loader))

        if val_auc > best_auc:
            best_auc   = val_auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if (epoch + 1) % 10 == 0:
            print(f'    epoch {epoch+1:3d}  loss={loss:.4f}  '
                  f'val_auc={val_auc:.5f}  best={best_auc:.5f}')

        if no_improve >= patience:
            print(f'    early stop at epoch {epoch+1}  best_auc={best_auc:.5f}')
            break

    model.load_state_dict(best_state)
    return best_auc


print('Training utilities ready.')

Training utilities ready.


## 5-Fold CV

Default FT-Transformer hyperparameters from the paper:
- `d_token=192`, `n_blocks=3`, `n_heads=8`, `d_ffn=256` (≈4/3 × 192)
- `attn_dropout=0.2`, `ffn_dropout=0.1`
- `lr=1e-4`, `weight_decay=1e-5`, `batch_size=1024`

In [5]:
FTT_PARAMS = dict(
    n_num=len(NUM_FEATURES),
    cardinalities=CARDINALITIES,
    d_token=192,
    n_blocks=3,
    n_heads=8,
    attn_dropout=0.2,
    ffn_dropout=0.1,
)
FIT_PARAMS = dict(
    batch_size=1024,
    max_epochs=100,
    patience=15,
    lr=1e-4,
    weight_decay=1e-5,
)

BASELINE_AUC = 0.95540  # CatBoost seed-avg

cv5  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof  = np.zeros(len(y))
test_folds = np.zeros((5, len(X_num_test)))

print(f'd_token={FTT_PARAMS["d_token"]}  n_blocks={FTT_PARAMS["n_blocks"]}  '
      f'n_heads={FTT_PARAMS["n_heads"]}  '
      f'lr={FIT_PARAMS["lr"]}  batch={FIT_PARAMS["batch_size"]}')
print()

for fold, (tr_idx, val_idx) in enumerate(cv5.split(X_num, y)):
    print(f'--- Fold {fold+1}/5 ---')

    torch.manual_seed(42 + fold)
    model = FTTransformer(**FTT_PARAMS).to(device)

    best_auc = fit_model(
        model,
        X_num[tr_idx], X_cat[tr_idx], y[tr_idx],
        X_num[val_idx], X_cat[val_idx], y[val_idx],
        **FIT_PARAMS
    )

    val_loader  = make_loader(X_num[val_idx],  X_cat[val_idx],  batch_size=4096)
    test_loader = make_loader(X_num_test, X_cat_test, batch_size=4096)

    oof[val_idx]      = predict_proba(model, val_loader)
    test_folds[fold]  = predict_proba(model, test_loader)

    print(f'Fold {fold+1} AUC: {roc_auc_score(y[val_idx], oof[val_idx]):.5f}')
    print()

cv_auc     = roc_auc_score(y, oof)
test_preds = test_folds.mean(axis=0)

print(f'FT-Transformer OOF AUC: {cv_auc:.5f}')
print(f'CatBoost seed-avg:       {BASELINE_AUC:.5f}')
print(f'Delta:                   {cv_auc - BASELINE_AUC:+.5f}')

d_token=192  n_blocks=3  n_heads=8  lr=0.0001  batch=1024

--- Fold 1/5 ---
    epoch  10  loss=0.2762  val_auc=0.95351  best=0.95355
    epoch  20  loss=0.2749  val_auc=0.95380  best=0.95380
    epoch  30  loss=0.2740  val_auc=0.95392  best=0.95392
    epoch  40  loss=0.2730  val_auc=0.95380  best=0.95392
    epoch  50  loss=0.2718  val_auc=0.95374  best=0.95396
    early stop at epoch 58  best_auc=0.95396
Fold 1 AUC: 0.95396

--- Fold 2/5 ---
    epoch  10  loss=0.2753  val_auc=0.95245  best=0.95252
    epoch  20  loss=0.2742  val_auc=0.95253  best=0.95271
    epoch  30  loss=0.2733  val_auc=0.95299  best=0.95299
    epoch  40  loss=0.2724  val_auc=0.95304  best=0.95304
    epoch  50  loss=0.2713  val_auc=0.95312  best=0.95314
    early stop at epoch 56  best_auc=0.95314
Fold 2 AUC: 0.95314

--- Fold 3/5 ---
    epoch  10  loss=0.2758  val_auc=0.95332  best=0.95339
    epoch  20  loss=0.2746  val_auc=0.95357  best=0.95366
    epoch  30  loss=0.2739  val_auc=0.95381  best=0.95381
    

## Results & Submission

In [6]:
import matplotlib.pyplot as plt

# Save OOF & test predictions for ensemble use
np.save('submissions/oof_ftt.npy',  oof)
np.save('submissions/test_ftt.npy', test_preds)

# Submission file
label = 'ftt_d192_b3'
fname = f'submissions/{label}.csv'
sub   = ss.copy()
sub['Heart Disease'] = test_preds
sub.to_csv(fname, index=False)
print(f'Saved: {fname}')

desc = f'{label} | cv_auc={cv_auc:.4f}'

ON_KAGGLE = Path('/kaggle/working').exists()
if ON_KAGGLE:
    print(f'Submit: kaggle competitions submit -c playground-series-s6e2 -f {fname} -m "{desc}"')
else:
    r = subprocess.run(
        ['kaggle', 'competitions', 'submit', '-c', 'playground-series-s6e2',
         '-f', fname, '-m', desc],
        capture_output=True, text=True
    )
    status = 'submitted' if 'successfully' in r.stdout.lower() else r.stdout.strip()[:100]
    print(f'{label}: {status}')
    print(f'desc: {desc}')

Saved: submissions/ftt_d192_b3.csv
ftt_d192_b3: submitted
desc: ftt_d192_b3 | cv_auc=0.9536


## GBT + FT-Transformer Ensemble

SLSQP-optimised weights on OOF predictions. Only submit if ensemble beats CatBoost seed-avg.

In [7]:
from scipy.optimize import minimize

oof_files  = {'catboost': 'submissions/oof_cat.npy',
              'lgbm':     'submissions/oof_lgb.npy',
              'xgb':      'submissions/oof_xgb.npy'}
test_files = {'catboost': 'submissions/test_cat.npy',
              'lgbm':     'submissions/test_lgb.npy',
              'xgb':      'submissions/test_xgb.npy'}

available = {k: Path(v).exists() for k, v in oof_files.items()}
print('GBT OOF files available:', available)

if all(available.values()):
    oofs  = {k: np.load(v) for k, v in oof_files.items()}
    tests = {k: np.load(v) for k, v in test_files.items()}
    oofs['ftt']  = oof
    tests['ftt'] = test_preds

    names    = list(oofs.keys())
    oof_list = list(oofs.values())

    def neg_auc(w):
        return -roc_auc_score(y, sum(wi * o for wi, o in zip(w, oof_list)))

    res = minimize(
        neg_auc, x0=[0.25] * len(names),
        method='SLSQP',
        bounds=[(0, 1)] * len(names),
        constraints={'type': 'eq', 'fun': lambda w: w.sum() - 1}
    )
    opt_auc = -res.fun
    print('\nSLSQP ensemble weights (GBTs + FT-Transformer):')
    for n, w in zip(names, res.x):
        print(f'  {n:<12} {w:.4f}')
    print(f'  OOF AUC = {opt_auc:.5f}  (vs baseline {BASELINE_AUC:.5f})')

    if opt_auc > BASELINE_AUC:
        pred = sum(w * tests[n] for w, n in zip(res.x, names))
        fname2 = 'submissions/ensemble_gbt_ftt.csv'
        desc2  = f'ensemble_gbt_ftt | cv_auc={opt_auc:.4f}'
        sub2   = ss.copy()
        sub2['Heart Disease'] = pred
        sub2.to_csv(fname2, index=False)
        if not ON_KAGGLE:
            r2 = subprocess.run(
                ['kaggle', 'competitions', 'submit', '-c', 'playground-series-s6e2',
                 '-f', fname2, '-m', desc2],
                capture_output=True, text=True
            )
            print(f'\nensemble_gbt_ftt: {"submitted" if "successfully" in r2.stdout.lower() else r2.stdout[:80]}')
    else:
        print('FT-Transformer did not improve ensemble — skipping submission.')
else:
    print('Run s6e2-gradient-boosting.ipynb first to generate GBT OOF files.')

GBT OOF files available: {'catboost': True, 'lgbm': True, 'xgb': True}

SLSQP ensemble weights (GBTs + FT-Transformer):
  catboost     0.2495
  lgbm         0.2502
  xgb          0.2509
  ftt          0.2495
  OOF AUC = 0.95519  (vs baseline 0.95540)
FT-Transformer did not improve ensemble — skipping submission.
